# B2-024 — Session 6: Mixture-Function Parameter Regression

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

The last task family asks you to read the parameters of a function off its samples.
Each function is a sum of three Gaussian bumps; you see it at 32 points, with noise, and must report the nine numbers that made it.
The difficulty is not the fitting alone: the three bumps have no natural order, so "the" parameter vector is not unique.
This session teaches mixture functions, identifiability and canonical ordering, permutation-invariant losses and metrics, a least-squares baseline by gradient descent (`book1:gradient-descent`, `book1:learning-rate`), a learned regressor that amortizes the fit, and the hybrid that combines them.

## 1. Mixture functions

A **three-bump mixture function** is
$$f(x;\theta)=\sum_{k=1}^{3} w_k\exp\!\Big(-\frac{(x-c_k)^2}{2s_k^2}\Big),$$
with **weights** $w_k>0$ (peak heights), **centers** $c_k$, and **widths** $s_k>0$.
Each bump has the shape of the `book1:gaussian-distribution` density without its normalizing constant, so $w_k$ is the bump's height at its center, not its area.

**The `mixture` dataset.** For each row the generator draws $w_k\sim U[0.5,1.5]$, $s_k\sim U[0.04,0.09]$, and three centers in $[0.1,0.9]$, redrawing until the smallest gap between sorted centers is at least $3\times$ the largest width, so the bumps are visibly separate.
It samples $f$ at the 32 equally spaced points `capstone_data.MIXTURE_X` $=0, \tfrac1{31},\dots,1$, adds independent $N(0,0.02^2)$ noise, and rounds to four decimals.
The targets are the nine parameters in the **canonical layout**
$$\theta=(w_1,c_1,s_1,\;w_2,c_2,s_2,\;w_3,c_3,s_3),\qquad c_1<c_2<c_3,$$
one component per triple, components sorted by center.

**Vectorized evaluation.** With `weights`, `centers`, `widths` of shape `(N, 3)` and `x` of shape `(P,)`, broadcasting gives all values at once: `(w[:, None, :] * torch.exp(-(x[None, :, None] - c[:, None, :]) ** 2 / (2 * s[:, None, :] ** 2))).sum(-1)` has shape `(N, P)`.
Practice p11 implements this as `mixture_function(x, weights, centers, widths)`.

**Sampling resolution.** The spacing $1/31\approx0.032$ is comparable to the smallest width $0.04$, so a narrow bump is seen at only three or four points.
That is enough to locate it, but noise on those few points limits how precisely its width can be read.

**Checkpoint 1A.** What is $f(c_2;\theta)$ approximately, when the bumps are well separated?

**Checkpoint 1B.** Give the shape of the output of the broadcast expression for $N=200$ functions and $P=32$ points, and the shape of the intermediate before `.sum(-1)`.

**Collected answers.** About $w_2$: bump 2 contributes $w_2e^0=w_2$ and the others contribute almost nothing, since $c_2$ is at least three of their widths away (each contributes at most $w_ke^{-9/2}\approx0.011w_k$). Output `(200, 32)`; intermediate `(200, 32, 3)`.

## 2. Identifiability, label switching, and canonical order

Relabelling the bumps does not change the function: for any permutation $\pi$ of $\{1,2,3\}$,
$$\sum_k w_{\pi(k)}\exp\!\Big(-\frac{(x-c_{\pi(k)})^2}{2s_{\pi(k)}^2}\Big)=\sum_k w_k\exp\!\Big(-\frac{(x-c_k)^2}{2s_k^2}\Big),$$
because a finite sum does not depend on the order of its terms.
So six different parameter vectors describe every function: the parameters are **not identifiable** as an ordered vector, only as an unordered set of three (weight, center, width) triples.
This is **label switching**.

**Why it matters for learning.** Suppose a network is trained with plain MSE against targets listed in the generator's *random* order.
For the same input function, the target is sometimes $(\text{bump A},\text{bump B},\ldots)$ and sometimes $(\text{bump B},\text{bump A},\ldots)$.
MSE is minimized by the average of the possible targets, so the network learns to predict every center near the *average* center: a collapsed, useless answer.

**Canonicalization** removes the ambiguity by choosing one representative of each set: **sort the components by center**.
When the centers are strictly distinct (here they differ by at least $3\times0.04=0.12$), there is exactly one ordering with $c_1<c_2<c_3$, so every function has exactly one canonical vector, and two parameter vectors describe the same mixture exactly when their canonical forms coincide.
Practice p14 proves both facts; p11 implements `canonicalize(params)`, which reorders the three triples of each row by center (a stable sort, keeping each triple intact).

The canonical layout turns an unordered-set problem into an ordinary regression target: the network always sees the leftmost bump first.

**Checkpoint 2A.** How many ordered parameter vectors describe one three-bump function with distinct centers?

**Checkpoint 2B.** Why does sorting by weight not give a canonical form for this dataset?

**Collected answers.** $3!=6$. Weights can tie (two bumps of equal height), and then the sorted order is not unique; centers are guaranteed distinct (separated by at least $0.12$), so sorting by center always gives one order.

## 3. Permutation-invariant losses and evaluation

A loss or metric $L(\hat\theta,\theta)$ is **permutation-invariant** if relabelling the components of either argument does not change it.
The direct way to build one is to minimize over relabellings:
$$L_{\text{PI}}(\hat\theta,\theta)=\min_{\pi\in S_3}\ \frac19\sum_{k=1}^{3}\Big(|\hat w_{\pi(k)}-w_k|+|\hat c_{\pi(k)}-c_k|+|\hat s_{\pi(k)}-s_k|\Big).$$
This unit's mixture metric is exactly this (`capstone_data.perm_invariant_error_per_example`): for each example, the smallest mean absolute difference over the nine parameters across the six ways of matching predicted components to true ones; the dataset score is the mean over examples.
With $K=3$ there are only six permutations, so computing all six is cheap.

**Canonicalize, then compare.** If both vectors are canonical, comparing them in order is usually the same as the minimum, because the best matching pairs the leftmost predicted bump with the leftmost true bump.
It can differ when a prediction's centers are badly wrong (for example two predicted bumps on the same true bump), so the official metric takes the minimum explicitly.

**Training losses.** Two good choices:

* train on canonical targets with ordinary MSE (the network learns to output canonical order), or
* train with a min-over-permutations loss (gradients flow through the best matching).

Either is permutation-invariant *as a procedure*; a plain MSE against non-canonical targets is not, and collapses as in Section 2.
Practice p05 asks you to recognize which loss is invariant; p14 proves the canonical form is unique.

**Scale.** Weights are around $1$, centers around $0.5$, widths around $0.06$, so an absolute error averages unlike quantities: a width error of $0.01$ is large relative to a width but small in the average.
This is a property of the stated metric, and you optimize the metric you are given.

**Checkpoint 3A.** Is $(\sum_k\hat w_k-\sum_kw_k)^2$ a permutation-invariant loss? Is it a good one?

**Checkpoint 3B.** A prediction is exactly the true parameter set, listed with its components in the order (rightmost, leftmost, middle). What are its in-order error against the canonical target and its permutation-invariant error?

**Collected answers.** It is invariant (sums ignore order) but useless alone: it is zero for every parameter vector with the right total weight, so it does not identify centers or widths. The in-order error is positive (it compares the rightmost bump with the leftmost, and so on); the permutation-invariant error is exactly $0$, because one of the six matchings pairs every predicted triple with its identical true triple.

## 4. A least-squares baseline by gradient descent

For one function, the **least-squares fit** minimizes the mean squared residual over the 32 sampled points,
$$\ell(\theta)=\frac{1}{32}\sum_{p=1}^{32}\big(f(x_p;\theta)-y_p\big)^2,$$
a nonlinear `book1:mse-loss` in $\theta$.
There is no closed form, so we descend its gradient (`book1:gradient-descent`) with Adam.

**Parameterization.** Widths must stay positive, so optimize $\log s_k$ and use $s_k=\exp(\log s_k)$; weights and centers are optimized directly.
**All functions at once.** Summing $\ell$ over a batch of functions, $\sum_n\ell_n(\theta_n)$, couples nothing: the gradient with respect to $\theta_n$ depends only on function $n$, and Adam's update is elementwise, so one batched optimizer is exactly $N$ independent fits.
The committed baseline (`capstone_data.baseline_score("mixture")`) runs Adam with learning rate $0.02$ (`book1:learning-rate`) for 500 steps from a **fixed initialization** for every function: centers $(0.25, 0.5, 0.75)$, weights all $1$, widths all equal to the median training width.
At the frozen seed its test error is $0.0091$.

**Local minima.** $\ell$ is not convex.
If a true bump is far from every initial center, the gradient pulling a model bump toward it is tiny (the model bump's Gaussian tail barely overlaps it), and the fit can settle with one model bump covering two true bumps, or a bump stuck between them.
Most functions are fitted well; a few are fitted badly, which inflates the mean error and widens its bootstrap interval (Session 3).

**Where a learned model helps.** A good *starting point* removes the local-minimum problem: a few LS steps from near the answer converge to the nearby minimum.

**Checkpoint 4A.** Why optimize $\log s$ instead of $s$?

**Checkpoint 4B.** Why is one Adam optimizer over a batch of 200 functions equivalent to 200 separate fits when the loss is a sum over functions?

**Collected answers.** So that every update keeps $s=\exp(\log s)>0$ without clipping, and so that steps are relative (multiplicative) in $s$, which suits a scale parameter. Each function's loss depends only on its own parameters, so the gradient for function $n$ is unaffected by the others, and Adam's moments and update are computed elementwise per parameter.

## 5. A learned regressor that amortizes the fit

An **amortized regressor** learns the map from the 32 sampled values to the nine canonical parameters directly, from the 800 training functions.
Like the learned inverse of Session 5, it costs one training run and then one forward pass per new function.

**The model.** An MLP `32 → 128 → 128 → 9` (`book1:mlp-architecture`), whose `forward` takes the raw `(B, 32)` values exactly as `train_rows` returns them and standardizes them with training-set statistics stored as registered buffers, trained with MSE on the **canonical** targets by full-batch Adam (`book1:trained-mlp`).
Parameter count: $(32\cdot128+128)+(128\cdot128+128)+(128\cdot9+9)=4224+16512+1161=21{,}897$.

**An honest comparison.** On this dataset, the raw regressor's validation error is about $0.024$, while the per-function least-squares baseline's is about $0.009$: **the regressor loses**.
That is not a failure of the method; it is what amortization trades away.
A regressor must be accurate for *every* function at once with a fixed set of weights learned from 800 examples; least squares spends 500 optimizer steps on *each* function and can reach the precision that function's own data allows.
The regressor's errors are smooth and moderate; the baseline's are tiny for most functions and large for a few (local minima).

**Report both.** In Practice p20 you report the raw regressor's validation error beside the pipeline's result.
Hiding a weak component because the final pipeline is good makes the writeup less useful: the reader cannot tell where the improvement came from.

**Checkpoint 5A.** Count the parameters of the first layer `Linear(32, 128)`.

**Checkpoint 5B.** Why can per-function least squares be more precise than an amortized regressor, and where is it weaker?

**Collected answers.** $32\cdot128+128=4224$. It optimizes each function's own parameters against its own data with many steps, so it can reach the precision the data allow; it is weaker when it starts far from the answer and gets stuck in a local minimum, which the regressor does not suffer from.

## 6. The hybrid pipeline and the open-ended workflow

The two methods fail in complementary ways, so combine them: **predict, then refine**.

1. The regressor predicts canonical parameters $\hat\theta^{(0)}$ for each function (one forward pass).
2. Clamp each predicted width to at least $10^{-3}$ (so its logarithm exists), then run least squares on that function's own 32 values, starting from $\hat\theta^{(0)}$: Adam on $(w, c, \log s)$ with learning rate $0.02$, for 100 steps.
3. Canonicalize the refined parameters.

The regressor puts every bump near its true center, so refinement starts in the right basin and converges quickly; 100 steps from a good start do better than 500 steps from a fixed start.
At the frozen seed the pipeline's test error is about half the baseline's.

**Counting the cost.** Refinement uses optimizer steps at prediction time: 100 steps for the whole batch of functions (one batched optimizer).
In the mini-competition (p27) every optimizer step, training and refinement alike, counts against `StepBudget(2000)`: $1{,}500+100=1{,}600$ steps fit.

**The open-ended workflow for parameter regression.**

1. Canonicalize targets; implement the permutation-invariant metric and test it on permuted copies.
2. Compute the committed LS baseline under the protocol.
3. Train the regressor on training functions; choose its settings on validation.
4. Add refinement; choose its step count on validation.
5. Report the raw regressor and the pipeline on validation, then score the pipeline **once** on the locked test set, with its bootstrap interval and the baseline's.

**Checkpoint 6A.** Why clamp predicted widths before refinement?

**Checkpoint 6B.** In p27, a solution trains the regressor for 1,800 steps and refines for 300. Does it fit the budget?

**Collected answers.** Refinement optimizes $\log s$; a predicted width that is zero or negative has no logarithm, so it must be made positive first. No: $1{,}800+300=2{,}100>2{,}000$; the 2,001st step raises `StepBudgetExceeded`.

## 7. Worked laboratory: label switching, a stuck fit, and predict-then-refine

The laboratory demonstrates the three ideas on validation rows only (no test call).
First it shows that a permuted parameter vector produces the identical function and that the permutation-invariant error of a permuted copy is zero while the naive in-order error is not.
Then it runs the fixed-initialization least-squares baseline on the validation functions and prints the worst-fitted one's true and fitted centers.
Finally it trains a small regressor for 300 steps (a reduced version of p20's 1,500), refines its predictions for 100 steps, and compares raw, refined, and baseline validation errors.

**Checkpoint 7A.** In the worst-fitted example, what does the relation between true and fitted centers suggest happened?

**Checkpoint 7B.** Which two printed numbers show that refinement helps, and on which split are they computed?

**Collected answers.** A fitted bump settled between two true bumps or covered two at once: a local minimum of the least-squares loss reached from the fixed initialization. The raw regressor's and the refined pipeline's mean permutation-invariant errors, on the validation rows.

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

x_pts = capstone_data.MIXTURE_X
val_f, val_theta = capstone_data.val_rows("mixture")
train_f, train_theta = capstone_data.train_rows("mixture")

theta = val_theta[:1]                                    # one canonical vector (1, 9)
swapped = theta.reshape(1, 3, 3)[:, [2, 0, 1]].reshape(1, 9)
same = torch.allclose(capstone_data.mixture_values(x_pts, theta), capstone_data.mixture_values(x_pts, swapped),
                      atol=1e-6, rtol=1e-6)
print("permuted parameters give the same function:", same)
print("naive in-order error of the permuted copy:", round((swapped - theta).abs().mean().item(), 4),
      "| permutation-invariant error:", capstone_data.perm_invariant_error_per_example(swapped, theta).item())

baseline_fit = capstone_data.mixture_ls_fit(val_f, capstone_data.mixture_baseline_init(val_f.shape[0]), steps=500)
base_err = capstone_data.perm_invariant_error_per_example(baseline_fit, val_theta)
worst = int(base_err.argmax())
print("baseline validation error", round(base_err.mean().item(), 4), "| median", round(base_err.median().item(), 4))
print("worst example", worst, "true centers", [round(v, 3) for v in val_theta[worst].reshape(3, 3)[:, 1].tolist()],
      "fitted centers", [round(v, 3) for v in baseline_fit[worst].reshape(3, 3)[:, 1].tolist()])

In [ ]:
class Regressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("mu", train_f.mean(dim=0))
        self.register_buffer("sd", train_f.std(dim=0))
        self.net = nn.Sequential(nn.Linear(32, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 9))

    def forward(self, values):
        return self.net((values - self.mu) / self.sd)

torch.manual_seed(SEED)
regressor = Regressor()
optimizer = torch.optim.Adam(regressor.parameters(), lr=1e-3)
for _ in range(300):
    optimizer.zero_grad(set_to_none=True)
    F.mse_loss(regressor(train_f), train_theta).backward()
    optimizer.step()
with torch.no_grad():
    raw = capstone_data.canonical_sort(regressor(val_f))
start = raw.reshape(-1, 3, 3).clone()
start[..., 2] = start[..., 2].clamp_min(1e-3)
refined = capstone_data.mixture_ls_fit(val_f, start.reshape(-1, 9), steps=100)
for name, pred in (("raw regressor (300 steps)", raw), ("regressor + 100 LS steps", refined), ("LS baseline (500 steps)", baseline_fit)):
    print(f"{name}: validation error {capstone_data.perm_invariant_error_per_example(pred, val_theta).mean().item():.4f}")

## 8. Common pitfalls and Exam connections

**Common pitfalls (each with its fix).**

1. *MSE against non-canonical targets.* Fix: canonicalize targets (or use a min-over-permutations loss).
2. *Sorting by weight or width.* Fix: sort by center, which is guaranteed distinct.
3. *Sorting the nine numbers independently.* Fix: reorder whole (weight, center, width) triples.
4. *Optimizing widths directly.* Fix: optimize $\log s$; clamp before taking the log.
5. *Evaluating without the permutation minimum.* Fix: use the official metric.
6. *Hiding the raw regressor.* Fix: report raw and refined numbers on validation.
7. *Refinement steps not counted.* Fix: wrap the refinement optimizer in the same `StepBudget`.
8. *Choosing the number of refinement steps on the test set.* Fix: validation, then one test call.

**Exam connections.** p05 identifies a permutation-invariant loss; p11 implements `mixture_function` and `canonicalize`; p14 proves invariance under permutation and uniqueness of the center-sorted form; p20 trains the regressor, refines with 100 LS steps, certifies the pipeline against the fixed-initialization LS baseline, and reports the raw regressor honestly; p27 is the open-ended mixture mini-competition with a step budget and the four-part writeup.

**Checkpoint 8A.** A `canonicalize` sorts the vector `(w1,c1,s1,w2,c2,s2,w3,c3,s3)` with `torch.sort` over all nine entries. What goes wrong?

**Checkpoint 8B.** Which practice requires reporting a component that loses to the baseline?

**Collected answers.** It mixes weights, centers, and widths into one sorted list and breaks the triples apart, producing a vector that describes a different function; sort the triples by their center instead. p20 (the raw regressor's validation error).

## 9. Forward-only deeper topic: set prediction, EM, and unknown K

Forward-only deeper topic.
Object detectors that predict a set of boxes face the same label switching; **set-prediction losses** match predictions to targets with the Hungarian algorithm, which scales to many components where trying all $K!$ permutations does not.
When the data are samples from a mixture *density* rather than values of a mixture *function*, **expectation–maximization** fits weights, means, and variances by alternating soft assignments and weighted averages, a soft relative of `book1:lloyd-algorithm`.
When the number of components is unknown, it can be chosen by validation error or an information criterion, or predicted by the network together with the parameters.
None of these are prerequisites, permitted shortcuts, or assessed APIs in B2-024.

This final section is forward-only and not assessed.

**Checkpoint 9A.** Why does trying all permutations not scale to $K=20$ components?

**Checkpoint 9B.** Which Book 1 algorithm is EM a soft version of?

**Collected answers.** $20!$ is about $2.4\times10^{18}$ permutations; the Hungarian algorithm finds the best matching in polynomial time. Lloyd's algorithm for k-means (hard assignments and plain averages).